In [ ]:
import os
import kagglehub
from pathlib import Path

# 1. Download dataset
path = kagglehub.dataset_download("bhamretejas/civic-problem")
print("Path to dataset files:", path)

# 2. Inspect root directory
root_path = Path(path)
print("\n--- Directory Content ---")
for item in root_path.iterdir():
    print(f"[{'DIR' if item.is_dir() else 'FILE'}] {item.name}")

# Check for subdirectories and image counts
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
subdirs = [d for d in root_path.iterdir() if d.is_dir()]

print("\n--- Class / Folder Analysis ---")
for d in subdirs:
    images = [f for f in d.rglob("*") if f.suffix.lower() in image_extensions]
    print(f"Directory '{d.name}': {len(images)} images found")

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from pathlib import Path
import kagglehub

# 1. Hardware verification
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if str(device) == "cpu":
    print("Warning: GPU not active. Enable GPU in Kaggle sidebar (Notebook options -> Accelerator -> GPU T4).")

# 2. Dataset resolution
dataset_path = kagglehub.dataset_download("bhamretejas/civic-problem")
root_path = Path(dataset_path)

# Locate folder containing category subfolders
data_dir = root_path
train_subdir = root_path / "train"
if train_subdir.exists() and train_subdir.is_dir():
    data_dir = train_subdir

print(f"Loading dataset from: {data_dir}")

# 3. Data Transformations
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. Load & Split Dataset
full_dataset = datasets.ImageFolder(root=str(data_dir), transform=train_transforms)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Identified {num_classes} classes: {class_names}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_data, val_data = random_split(
    full_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply validation transforms to val_data
val_data.dataset.transform = val_transforms

# num_workers=0 avoids Kaggle thread deadlock
train_loader = DataLoader(
    train_data, 
    batch_size=32, 
    shuffle=True, 
    num_workers=0, 
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_data, 
    batch_size=32, 
    shuffle=False, 
    num_workers=0, 
    pin_memory=(device.type == "cuda")
)

# 5. Model Architecture (EfficientNet-B0 transfer learning)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze feature extractor for rapid training
for param in model.features.parameters():
    param.requires_grad = False

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, num_classes)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

# 6. Training Loop
EPOCHS = 10
best_acc = 0.0

print(f"\nStarting training ({EPOCHS} epochs)...\n")

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)
        
    train_loss = running_loss / total
    train_acc = correct / total

    # Validation pass
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data).item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] - Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/best_civic_model.pth")

print(f"\nTraining completed. Top Validation Accuracy: {best_acc*100:.2f}%")
print("Weights saved: /kaggle/working/best_civic_model.pth")

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import requests
from pathlib import Path
import kagglehub

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 1. Resolve dataset directory dynamically
dataset_path = kagglehub.dataset_download("bhamretejas/civic-problem")
data_dir = Path(dataset_path) / "civic-issue-dataset"
if not data_dir.exists():
    data_dir = Path(dataset_path)

# Ensure class_names and num_classes exist
if 'class_names' not in globals():
    class_names = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
num_classes = len(class_names)

checkpoint_path = "/kaggle/working/best_civic_model.pth"

# 2. Ensure model file exists; if model is still resident in memory, persist it
if not os.path.exists(checkpoint_path):
    if 'model' in globals():
        print("Persisting in-memory model to disk...")
        torch.save(model.state_dict(), checkpoint_path)
    else:
        raise FileNotFoundError("Weights file not found and 'model' is not in memory. Re-run training.")

# 3. Build inference model
infer_model = models.efficientnet_b0()
in_features = infer_model.classifier[1].in_features
infer_model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, num_classes)
)
infer_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
infer_model.to(device)
infer_model.eval()

# 4. Evaluation Transform
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 5. Prediction & Priority Function
def predict_civic_issue(image_path: str, user_lat: float = None, user_lon: float = None):
    img = Image.open(image_path).convert("RGB")
    tensor_img = eval_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = infer_model(tensor_img)
        probs = torch.softmax(logits, dim=1)[0]
        confidence, pred_idx = torch.max(probs, 0)
        predicted_category = class_names[pred_idx.item()]

    # Critical facility check (Hospital, School, College)
    is_sensitive_area = False
    if user_lat is not None and user_lon is not None:
        try:
            query = f"""
            [out:json][timeout:4];
            (
              node["amenity"~"hospital|school|college|university"](around:400,{user_lat},{user_lon});
              way["amenity"~"hospital|school|college|university"](around:400,{user_lat},{user_lon});
            );
            out count;
            """
            res = requests.post("https://overpass-api.de/api/interpreter", data={'data': query}, timeout=4)
            if res.status_code == 200:
                count_val = int(res.json().get('elements', [{}])[0].get('tags', {}).get('total', '0'))
                is_sensitive_area = (count_val > 0)
        except Exception:
            is_sensitive_area = False

    # Urgency weights
    urgency_weights = {
        "drainage": 4.5,
        "water": 4.0,
        "pothole": 3.5,
        "road": 3.5,
        "garbage": 3.0,
        "streetlight": 2.0
    }
    
    base_urgency = 3.0
    for key, weight in urgency_weights.items():
        if key in predicted_category.lower():
            base_urgency = weight
            break

    multiplier = 1.5 if is_sensitive_area else 1.0
    total_score = base_urgency * multiplier

    if total_score >= 5.5:
        priority = "High"
    elif total_score >= 3.5:
        priority = "Medium"
    else:
        priority = "Low"

    return {
        "category": predicted_category,
        "confidence": f"{confidence.item() * 100:.2f}%",
        "near_hospital_or_school": is_sensitive_area,
        "priority": priority
    }

# 6. Test on image
sample_image = next(data_dir.rglob("*.jpg"))
result = predict_civic_issue(str(sample_image), user_lat=19.0760, user_lon=72.8777)

print("--- Prediction Result ---")
for key, val in result.items():
    print(f"{key}: {val}")

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import json
import torch
import onnx
import onnxruntime as ort

# 1. Save class names
labels_path = "/kaggle/working/classes.json"
with open(labels_path, "w") as f:
    json.dump(class_names, f)
print(f"Classes saved: {class_names}")

# 2. Export using classic exporter (dynamo=False)
onnx_file_path = "/kaggle/working/civic_pulse_model.onnx"
dummy_input = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    infer_model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

# 3. Validate
onnx_model = onnx.load(onnx_file_path)
onnx.checker.check_model(onnx_model)
print(f"ONNX model verified and saved at: {onnx_file_path}")

# 4. Test run with ONNX Runtime
ort_session = ort.InferenceSession(onnx_file_path, providers=["CPUExecutionProvider"])
test_np = dummy_input.cpu().numpy()
ort_outputs = ort_session.run(None, {"input": test_np})[0]

print(f"ONNX Runtime test output shape: {ort_outputs.shape}")
print("Verification complete.")

In [ ]:
from pathlib import Path
import kagglehub

dataset_path = kagglehub.dataset_download("bhamretejas/civic-problem")
root = Path(dataset_path)

print("--- Directory Tree ---")
for p in root.rglob("*"):
    if p.is_dir():
        print(f"[DIR]  {p.relative_to(root)}")
    elif p.suffix.lower() in [".csv", ".json", ".txt", ".yaml", ".yml"]:
        print(f"[FILE] {p.relative_to(root)} (Size: {p.stat().st_size} bytes)")

In [ ]:
from pathlib import Path
import kagglehub

dataset_path = kagglehub.dataset_download("bhamretejas/civic-problem")
root = Path(dataset_path)

print("--- Checking Dataset Structure ---")
for p in root.rglob("*"):
    if p.is_dir():
        print(f"[DIR]  {p.relative_to(root)}")
    elif p.suffix.lower() in [".csv", ".json", ".txt", ".yaml", ".yml"]:
        print(f"[FILE] {p.relative_to(root)} (Size: {p.stat().st_size} bytes)")

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from pathlib import Path
import kagglehub

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# 1. Direct path to category folders
dataset_path = kagglehub.dataset_download("bhamretejas/civic-problem")
real_data_dir = Path(dataset_path) / "civic-issue-dataset" / "images"

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Load dataset
full_dataset = datasets.ImageFolder(root=str(real_data_dir), transform=train_transforms)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"Loaded {num_classes} classes: {class_names}")

# Save class labels to disk
with open("/kaggle/working/classes.json", "w") as f:
    json.dump(class_names, f)

# 80/20 train/val split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(
    full_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)
val_data.dataset.transform = val_transforms

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=0, pin_memory=(device.type == "cuda"))

# 3. Model Architecture
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Fine-tune the top layers
for param in model.features[:-3].parameters():
    param.requires_grad = False

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, num_classes)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4, weight_decay=1e-2)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

# 4. Training (7 epochs is fast and reaches >90% accuracy)
EPOCHS = 7
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)
        
    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data).item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] -> Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/best_civic_model.pth")

print(f"\nTraining Complete! Best Accuracy: {best_acc*100:.2f}%")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
import json

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 1. Load labels & weights
with open("/kaggle/working/classes.json", "r") as f:
    saved_classes = json.load(f)

infer_model = models.efficientnet_b0()
infer_model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(infer_model.classifier[1].in_features, len(saved_classes))
)
infer_model.load_state_dict(torch.load("/kaggle/working/best_civic_model.pth", map_location=device))
infer_model.to(device)
infer_model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Priority Engine Function
def analyze_civic_image(image_path: str, is_near_hospital_or_school: bool = False):
    img = Image.open(image_path).convert("RGB")
    tensor_img = eval_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = infer_model(tensor_img)
        probs = torch.softmax(logits, dim=1)[0]
        conf, idx = torch.max(probs, 0)
        category = saved_classes[idx.item()]

    # If it is a normal road or working streetlight, no issue
    if category in ["road_normal", "streetlight_good"]:
        return {
            "category": category,
            "confidence": f"{conf.item() * 100:.2f}%",
            "issue_detected": False,
            "priority": "None (Normal State)"
        }

    # Base Urgency weights for civic hazards
    urgency_scores = {
        "open_manhole": 5.0,     # Fatal danger to pedestrians/vehicles
        "potholes": 3.5,         # Road vehicle hazard
        "garbage": 3.0,          # Health / hygiene hazard
        "streetlight_bad": 2.0   # Night safety hazard
    }
    base_urgency = urgency_scores.get(category, 3.0)

    # 1.5x multiplier for schools / hospitals / colleges
    location_multiplier = 1.5 if is_near_hospital_or_school else 1.0
    total_score = base_urgency * location_multiplier

    if total_score >= 5.0:
        priority = "High"
    elif total_score >= 3.0:
        priority = "Medium"
    else:
        priority = "Low"

    return {
        "category": category,
        "confidence": f"{conf.item() * 100:.2f}%",
        "issue_detected": True,
        "near_hospital_or_school": is_near_hospital_or_school,
        "priority": priority
    }

# 3. Test on 1 sample image
# Replace with any image path you upload: e.g. "/kaggle/working/my_test.jpg"
sample_img = next(Path(real_data_dir).rglob("*.jpg"))

result = analyze_civic_image(str(sample_img), is_near_hospital_or_school=True)

print("\n--- Civic Pulse Output ---")
for k, v in result.items():
    print(f"{k}: {v}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import torch

def analyze_civic_image_calibrated(image_path: str, is_near_hospital_or_school: bool = False):
    img = Image.open(image_path).convert("RGB")
    tensor_img = eval_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = infer_model(tensor_img)
        probs = torch.softmax(logits, dim=1)[0]
        conf, idx = torch.max(probs, 0)
        category = saved_classes[idx.item()]

    # Benign / normal conditions
    if category in ["road_normal", "streetlight_good"]:
        return {
            "category": category,
            "confidence": conf.item(),
            "issue_detected": False,
            "near_critical_zone": is_near_hospital_or_school,
            "priority": "None"
        }

    # Urgency weights (1.0 to 5.0)
    urgency_scores = {
        "open_manhole": 5.0,    # Immediate fall/fatal risk
        "potholes": 3.5,        # Vehicle damage/accident risk
        "garbage": 3.2,         # Health/sanitation hazard
        "streetlight_bad": 2.2  # Visibility/night safety
    }
    base_urgency = urgency_scores.get(category, 3.0)

    # Proximity Multiplier: 1.5x boost near schools, hospitals, or transit nodes
    multiplier = 1.5 if is_near_hospital_or_school else 1.0
    total_score = base_urgency * multiplier

    # Calibrated thresholds:
    # Score >= 4.5 -> High (e.g., Garbage near School: 3.2 * 1.5 = 4.8 -> High)
    # Score >= 3.0 -> Medium
    # Score < 3.0  -> Low
    if total_score >= 4.5:
        priority = "High"
    elif total_score >= 3.0:
        priority = "Medium"
    else:
        priority = "Low"

    return {
        "category": category,
        "confidence": conf.item(),
        "issue_detected": True,
        "near_critical_zone": is_near_hospital_or_school,
        "priority": priority
    }

# --- Visual Inspection Across Different Categories ---
test_categories = ["garbage", "open_manhole", "potholes", "streetlight_bad"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, cat in enumerate(test_categories):
    cat_folder = real_data_dir / cat
    sample_file = next(cat_folder.glob("*.jpg"), None) or next(cat_folder.glob("*.png"), None)
    
    if sample_file:
        # Evaluate with critical zone = True
        res = analyze_civic_image_calibrated(str(sample_file), is_near_hospital_or_school=True)
        
        raw_img = Image.open(sample_file)
        axes[i].imshow(raw_img)
        axes[i].axis("off")
        
        color = "red" if res["priority"] == "High" else "orange" if res["priority"] == "Medium" else "green"
        axes[i].set_title(
            f"Category: {res['category']}\nConf: {res['confidence']*100:.1f}%\nPriority: {res['priority']} (Near School/Hosp)",
            color=color,
            fontsize=11,
            fontweight="bold"
        )

plt.tight_layout()
plt.show()

In [ ]:
import json
import torch

# 1. Save label definitions
with open("/kaggle/working/classes.json", "w") as f:
    json.dump(saved_classes, f)

# 2. Export ONNX (using dynamo=False to avoid missing onnxscript errors)
onnx_file_path = "/kaggle/working/civic_pulse_model.onnx"
dummy_input = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    infer_model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

print(f"ONNX Model saved to: {onnx_file_path}")
print(f"Class count: {len(saved_classes)} -> {saved_classes}")

In [ ]:
import json
import numpy as np
import onnxruntime as ort
from PIL import Image

# 1. Load classes and initialize ONNX runtime session
with open("/kaggle/working/classes.json", "r") as f:
    class_names = json.load(f)

session = ort.InferenceSession("/kaggle/working/civic_pulse_model.onnx", providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

# 2. Image preprocessor matching ImageNet specs
def preprocess_image(image_path: str):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((224, 224))
    arr = np.array(img).astype(np.float32) / 255.0

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    arr = (arr - mean) / std

    # (H, W, C) -> (1, C, H, W)
    arr = np.transpose(arr, (2, 0, 1))
    return np.expand_dims(arr, axis=0)

# 3. Softmax implementation
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / np.sum(e_x, axis=1, keepdims=True)

# 4. End-to-end Civic Pulse analysis
def analyze_civic_pulse(image_path: str, near_hospital_or_school: bool = False):
    input_tensor = preprocess_image(image_path)
    outputs = session.run(None, {input_name: input_tensor})[0]
    probabilities = softmax(outputs)[0]

    top_idx = int(np.argmax(probabilities))
    category = class_names[top_idx]
    confidence = float(probabilities[top_idx])

    # Normal / Non-hazard state
    if category in ["road_normal", "streetlight_good"]:
        return {
            "category": category,
            "confidence": f"{confidence * 100:.2f}%",
            "issue_detected": False,
            "near_critical_facility": near_hospital_or_school,
            "priority": "None"
        }

    # Urgency scoring table
    base_urgencies = {
        "open_manhole": 5.0,    # Fall/fatality hazard
        "potholes": 3.5,        # Vehicle collision/accident hazard
        "garbage": 3.2,         # Public hygiene/epidemic hazard
        "streetlight_bad": 2.2  # Low-visibility hazard
    }
    base_score = base_urgencies.get(category, 3.0)

    # 1.5x Multiplier for proximity to schools, colleges, or hospitals
    multiplier = 1.5 if near_hospital_or_school else 1.0
    final_score = base_score * multiplier

    # Calibrated thresholds
    if final_score >= 4.5:
        priority = "High"
    elif final_score >= 3.0:
        priority = "Medium"
    else:
        priority = "Low"

    return {
        "category": category,
        "confidence": f"{confidence * 100:.2f}%",
        "issue_detected": True,
        "near_critical_facility": near_hospital_or_school,
        "priority": priority
    }

In [ ]:
from pathlib import Path

# Grab an image from the dataset or use your own custom file path
test_image_path = next(Path("/kaggle/input/datasets/bhamretejas/civic-problem/civic-issue-dataset/images/potholes").glob("*.jpg"))

output = analyze_civic_pulse(str(test_image_path), near_hospital_or_school=True)

print("--- Civic Pulse Output ---")
for key, value in output.items():
    print(f"{key}: {value}")

In [ ]:
import io
import json
import numpy as np
import onnxruntime as ort
import requests
from fastapi import FastAPI, File, Form, UploadFile
from PIL import Image

app = FastAPI(title="Civic Pulse ML Service")

# 1. Load ONNX model and class labels at startup
with open("classes.json", "r") as f:
    class_names = json.load(f)

session = ort.InferenceSession("civic_pulse_model.onnx", providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name


def check_critical_zone(lat: float, lon: float, radius_meters: int = 400) -> bool:
    """Queries OpenStreetMap Overpass API for schools, colleges, or hospitals."""
    query = f"""
    [out:json][timeout:3];
    (
      node["amenity"~"hospital|school|college|university"](around:{radius_meters},{lat},{lon});
      way["amenity"~"hospital|school|college|university"](around:{radius_meters},{lat},{lon});
    );
    out count;
    """
    try:
        res = requests.post(
            "https://overpass-api.de/api/interpreter",
            data={"data": query},
            timeout=3.5,
        )
        if res.status_code == 200:
            count = int(res.json().get("elements", [{}])[0].get("tags", {}).get("total", "0"))
            return count > 0
    except Exception:
        pass
    return False


def preprocess(image_bytes: bytes) -> np.ndarray:
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB").resize((224, 224))
    arr = np.array(img).astype(np.float32) / 255.0

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    arr = (arr - mean) / std

    arr = np.transpose(arr, (2, 0, 1))
    return np.expand_dims(arr, axis=0)


def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)


@app.post("/api/report-issue")
async def report_issue(
    file: UploadFile = File(...),
    latitude: float = Form(None),
    longitude: float = Form(None),
):
    # 1. Read and preprocess image
    image_bytes = await file.read()
    input_tensor = preprocess(image_bytes)

    # 2. ONNX Inference
    outputs = session.run(None, {input_name: input_tensor})[0]
    probs = softmax(outputs)[0]

    top_idx = int(np.argmax(probs))
    category = class_names[top_idx]
    confidence = float(probs[top_idx])

    # 3. Normal conditions
    if category in ["road_normal", "streetlight_good"]:
        return {
            "status": "success",
            "category": category,
            "confidence": round(confidence, 4),
            "issue_detected": False,
            "near_critical_facility": False,
            "priority": "None",
        }

    # 4. Check critical facility
    near_critical = False
    if latitude is not None and longitude is not None:
        near_critical = check_critical_zone(latitude, longitude)

    # 5. Priority engine
    urgency_scores = {
        "open_manhole": 5.0,
        "potholes": 3.5,
        "garbage": 3.2,
        "streetlight_bad": 2.2,
    }
    base = urgency_scores.get(category, 3.0)
    multiplier = 1.5 if near_critical else 1.0
    score = base * multiplier

    if score >= 4.5:
        priority = "High"
    elif score >= 3.0:
        priority = "Medium"
    else:
        priority = "Low"

    return {
        "status": "success",
        "category": category,
        "confidence": round(confidence, 4),
        "issue_detected": True,
        "near_critical_facility": near_critical,
        "priority": priority,
    }
    

In [ ]:
from IPython.display import FileLink

display(FileLink("civic_pulse_model.onnx"))
display(FileLink("classes.json"))
display(FileLink("best_civic_model.pth"))

In [ ]:
pip install fastapi uvicorn onnxruntime pillow numpy requests python-multipart

In [ ]:
%%writefile main.py
import io
import json
import numpy as np
import onnxruntime as ort
import requests
from fastapi import FastAPI, File, Form, UploadFile
from PIL import Image

app = FastAPI(title="Civic Pulse ML Service")

with open("/kaggle/working/classes.json", "r") as f:
    class_names = json.load(f)

session = ort.InferenceSession("/kaggle/working/civic_pulse_model.onnx", providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

def check_critical_zone(lat: float, lon: float, radius_meters: int = 400) -> bool:
    query = f"""
    [out:json][timeout:3];
    (
      node["amenity"~"hospital|school|college|university"](around:{radius_meters},{lat},{lon});
      way["amenity"~"hospital|school|college|university"](around:{radius_meters},{lat},{lon});
    );
    out count;
    """
    try:
        res = requests.post("https://overpass-api.de/api/interpreter", data={"data": query}, timeout=3.5)
        if res.status_code == 200:
            count = int(res.json().get("elements", [{}])[0].get("tags", {}).get("total", "0"))
            return count > 0
    except Exception:
        pass
    return False

def preprocess(image_bytes: bytes) -> np.ndarray:
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB").resize((224, 224))
    arr = np.array(img).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    arr = (arr - mean) / std
    arr = np.transpose(arr, (2, 0, 1))
    return np.expand_dims(arr, axis=0)

def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)

@app.post("/api/report-issue")
async def report_issue(
    file: UploadFile = File(...),
    latitude: float = Form(None),
    longitude: float = Form(None),
):
    image_bytes = await file.read()
    input_tensor = preprocess(image_bytes)

    outputs = session.run(None, {input_name: input_tensor})[0]
    probs = softmax(outputs)[0]

    top_idx = int(np.argmax(probs))
    category = class_names[top_idx]
    confidence = float(probs[top_idx])

    if category in ["road_normal", "streetlight_good"]:
        return {
            "status": "success",
            "category": category,
            "confidence": round(confidence, 4),
            "issue_detected": False,
            "near_critical_facility": False,
            "priority": "None",
        }

    near_critical = False
    if latitude is not None and longitude is not None:
        near_critical = check_critical_zone(latitude, longitude)

    urgency_scores = {
        "open_manhole": 5.0,
        "potholes": 3.5,
        "garbage": 3.2,
        "streetlight_bad": 2.2,
    }
    base = urgency_scores.get(category, 3.0)
    multiplier = 1.5 if near_critical else 1.0
    score = base * multiplier

    if score >= 4.5:
        priority = "High"
    elif score >= 3.0:
        priority = "Medium"
    else:
        priority = "Low"

    return {
        "status": "success",
        "category": category,
        "confidence": round(confidence, 4),
        "issue_detected": True,
        "near_critical_facility": near_critical,
        "priority": priority,
    }

In [ ]:
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio

In [ ]:
import threading
import uvicorn
from main import app

# Define server configuration
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

# Run server in a detached background thread
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

print("FastAPI server is running in background at http://127.0.0.1:8000")

In [ ]:
import requests
from pathlib import Path

# Pick a test image from your dataset
sample_image_path = next(Path("/kaggle/input/datasets/bhamretejas/civic-problem/civic-issue-dataset/images/potholes").glob("*.jpg"))

url = "http://127.0.0.1:8000/api/report-issue"

with open(sample_image_path, "rb") as img_file:
    files = {"file": ("test.jpg", img_file, "image/jpeg")}
    data = {
        "latitude": 19.0760,   # Sample coordinates
        "longitude": 72.8777
    }
    response = requests.post(url, files=files, data=data)

print("Status Code:", response.status_code)
print("Response JSON:")
print(response.json())

In [ ]:
from IPython.display import FileLink

display(FileLink("civic_pulse_model.onnx"))
display(FileLink("classes.json"))

In [ ]:
import os
from pathlib import Path

working_dir = Path("/kaggle/working")
files = list(working_dir.iterdir())

print("Files currently in /kaggle/working:")
for f in files:
    print(f" - {f.name} ({f.stat().st_size} bytes)")
    

In [ ]:
import json
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path
import kagglehub

# 1. Resolve classes from the dataset
dataset_path = kagglehub.dataset_download("bhamretejas/civic-problem")
real_data_dir = Path(dataset_path) / "civic-issue-dataset" / "images"
class_names = sorted([d.name for d in real_data_dir.iterdir() if d.is_dir()])
num_classes = len(class_names)

# Save classes.json
json_path = "/kaggle/working/classes.json"
with open(json_path, "w") as f:
    json.dump(class_names, f)
print(f"Saved {json_path}: {class_names}")

# 2. Build model structure
model = models.efficientnet_b0()
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, num_classes)
)

# Load checkpoint if it exists, otherwise use in-memory weights
ckpt_path = "/kaggle/working/best_civic_model.pth"
if Path(ckpt_path).exists():
    model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
    print("Loaded weights from best_civic_model.pth")
elif 'infer_model' in globals():
    model = infer_model.to("cpu")
    print("Using infer_model from active session memory")
elif 'model' in globals():
    model = model.to("cpu")
    print("Using model from active session memory")

model.eval()

# 3. Export to ONNX
onnx_path = "/kaggle/working/civic_pulse_model.onnx"
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print(f"Exported {onnx_path}")